In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd

from openai import OpenAI
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.text import tokenizer_from_json
from tensorflow.keras.preprocessing.sequence import pad_sequences

### Config

In [2]:
SHARED_DATASET_FILE = "../data/processed/cleaned_emotions_shared.csv"
SHARED_SPLIT_FILE   = "../data/splits/shared_split.json"

ML_METADATA_FILE    = "../models/ml_all_models_metadata.json"

DL_MODEL_FILE       = "../models/bilstm_model.keras"
TOKENIZER_FILE      = "../models/tokenizer.json"
DL_METADATA_FILE    = "../models/dl_metadata.json"

OUTPUT_FILE         = "../data/processed/hybrid_predictions.csv"
SUMMARY_FILE        = "../data/processed/final_summary.csv"
PER_CLASS_FILE      = "../data/processed/per_class_summary.csv"
CONFUSION_FILE      = "../data/processed/confusion_matrix_all.csv"

FEW_SHOT_K = 2
USE_QWEN_ONLY_WHEN_DISAGREE = False   # True = ให้ Qwen ทำงานเฉพาะตอน baseline ไม่แน่ใจ/ไม่ตรงกันใน logic อื่น
PRINT_SAMPLE_COUNT = 5

### Qwen via Ollama

In [3]:
client = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)

QWEN_MODEL = "qwen2.5:7b-instruct"
print("AI ready:", QWEN_MODEL)

AI ready: qwen2.5:7b-instruct


### Setup

In [4]:
os.makedirs("../data/processed", exist_ok=True)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

### Load shared data

In [5]:
df = pd.read_csv(SHARED_DATASET_FILE)

with open(SHARED_SPLIT_FILE, "r", encoding="utf-8") as f:
    split = json.load(f)

train_df = df[df["row_id"].isin(split["train_ids"])].copy()
test_df  = df[df["row_id"].isin(split["test_ids"])].copy()

VALID_LABELS = sorted(train_df["label"].unique().tolist())

print("Train size:", train_df.shape)
print("Test size :", test_df.shape)
print("Valid labels:", VALID_LABELS)

Train size: (1512, 3)
Test size : (324, 3)
Valid labels: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad']


### Few-shot bank

In [6]:
def build_few_shot_bank(train_data, k, valid_labels):
    bank = {}
    for label in valid_labels:
        subset = train_data[train_data["label"] == label]
        examples = subset["text"].sample(
            n=min(k, len(subset)),
            random_state=42
        ).tolist()
        bank[label] = examples
    return bank

print("\nBuilding few-shot bank...")
few_shot_bank = build_few_shot_bank(train_df, FEW_SHOT_K, VALID_LABELS)


Building few-shot bank...


### Ask Qwen

In [7]:
def ask_qwen(text: str, baseline_label: str) -> str:
    labels_str = ", ".join(VALID_LABELS)

    examples_text = ""
    for label, examples in few_shot_bank.items():
        for ex in examples:
            examples_text += f'- "{ex}" -> {label}\n'

    prompt = f"""You are an emotion classification assistant.

Below are some labeled examples:
{examples_text}

Task:
Read the text carefully and verify whether the baseline predicted label is correct.

Text: "{text}"
Baseline predicted label: "{baseline_label}"
Valid labels: {labels_str}

Rules:
1. If the baseline label matches the text meaning, keep it.
2. If the baseline label is not correct, replace it with the best label from the valid labels.
3. Reply with ONLY one label from the valid labels.
"""

    try:
        response = client.chat.completions.create(
            model=QWEN_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        result = response.choices[0].message.content.strip().lower()

        if result in VALID_LABELS:
            return result

        for label in VALID_LABELS:
            if label in result:
                return label

        return baseline_label

    except Exception as e:
        print(f"[Qwen error] {e} -> fallback to baseline")
        return baseline_label

### Run Qwen refinement

In [8]:
def run_hybrid(texts, baseline_preds, tag=""):
    hybrid_preds = []
    total = len(texts)

    print(f"\n[{tag}] Sending {total} samples to Qwen...")
    for i, (text, pred) in enumerate(zip(texts, baseline_preds), start=1):
        if i % 50 == 0 or i == total:
            print(f"Progress: {i}/{total}")
        hybrid_pred = ask_qwen(text, pred)
        hybrid_preds.append(hybrid_pred)

    return hybrid_preds

# =========================================================
# Evaluation helpers
# =========================================================
def get_summary_metrics(y_true, y_pred, model_name):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1_Macro": f1_score(y_true, y_pred, average="macro"),
        "F1_Weighted": f1_score(y_true, y_pred, average="weighted")
    }

def get_per_class_report(y_true, y_pred, model_name):
    report = classification_report(
        y_true,
        y_pred,
        labels=VALID_LABELS,
        output_dict=True,
        zero_division=0
    )

    rows = []
    for label in VALID_LABELS:
        rows.append({
            "Model": model_name,
            "Emotion": label,
            "Precision": report[label]["precision"],
            "Recall": report[label]["recall"],
            "F1_Score": report[label]["f1-score"],
            "Support": int(report[label]["support"])
        })
    return pd.DataFrame(rows)

def get_confusion_rows(y_true, y_pred, model_name):
    cm = confusion_matrix(y_true, y_pred, labels=VALID_LABELS)
    rows = []
    for true_idx, true_label in enumerate(VALID_LABELS):
        for pred_idx, pred_label in enumerate(VALID_LABELS):
            rows.append({
                "Model": model_name,
                "True_Label": true_label,
                "Pred_Label": pred_label,
                "Count": int(cm[true_idx, pred_idx])
            })
    return pd.DataFrame(rows)

def print_model_result(y_true, y_pred, model_name):
    print(f"\n===== {model_name} =====")
    print("Accuracy    :", accuracy_score(y_true, y_pred))
    print("F1 Macro    :", f1_score(y_true, y_pred, average="macro"))
    print("F1 Weighted :", f1_score(y_true, y_pred, average="weighted"))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, labels=VALID_LABELS, zero_division=0))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred, labels=VALID_LABELS))

def print_per_emotion_summary(per_class_df, model_name):
    print(f"\n===== Per-Emotion Summary: {model_name} =====")
    show_df = per_class_df[per_class_df["Model"] == model_name].copy()
    print(show_df[["Emotion", "Precision", "Recall", "F1_Score", "Support"]])

def print_prediction_distribution(y_pred, model_name):
    pred_series = pd.Series(y_pred)
    dist = pred_series.value_counts().reindex(VALID_LABELS, fill_value=0).reset_index()
    dist.columns = ["Emotion", "Predicted_Count"]
    print(f"\n===== Predicted Emotion Distribution: {model_name} =====")
    print(dist)

### Start containers

In [9]:
result_df = test_df[["row_id", "text", "label"]].copy()
summary_rows = []
per_class_frames = []
confusion_frames = []

y_true = test_df["label"]

### Load ML models

In [10]:
with open(ML_METADATA_FILE, "r", encoding="utf-8") as f:
    ml_metadata = json.load(f)

vectorizer = joblib.load(ml_metadata["vectorizer_file"])
X_test_ml = vectorizer.transform(test_df["text"])

ml_model_names = [
    "Naive Bayes",
    "Logistic Regression",
    "Random Forest",
    "Support Vector Machine"
]

for model_name in ml_model_names:
    model_path = ml_metadata[model_name]["model_path"]
    pred_col = ml_metadata[model_name]["prediction_column"]

    print(f"\nLoading ML model: {model_name}")
    model = joblib.load(model_path)

    baseline_preds = model.predict(X_test_ml)
    result_df[pred_col] = baseline_preds

    print_model_result(y_true, baseline_preds, model_name)
    print_prediction_distribution(baseline_preds, model_name)

    summary_rows.append(get_summary_metrics(y_true, baseline_preds, model_name))
    per_class_frames.append(get_per_class_report(y_true, baseline_preds, model_name))
    confusion_frames.append(get_confusion_rows(y_true, baseline_preds, model_name))

    hybrid_col = pred_col.replace("_pred", "_qwen_pred")
    hybrid_preds = run_hybrid(
        texts=test_df["text"].tolist(),
        baseline_preds=baseline_preds,
        tag=model_name
    )
    result_df[hybrid_col] = hybrid_preds

    hybrid_name = f"{model_name} + Qwen"
    print_model_result(y_true, hybrid_preds, hybrid_name)
    print_prediction_distribution(hybrid_preds, hybrid_name)

    summary_rows.append(get_summary_metrics(y_true, hybrid_preds, hybrid_name))
    per_class_frames.append(get_per_class_report(y_true, hybrid_preds, hybrid_name))
    confusion_frames.append(get_confusion_rows(y_true, hybrid_preds, hybrid_name))



Loading ML model: Naive Bayes

===== Naive Bayes =====
Accuracy    : 0.8827160493827161
F1 Macro    : 0.8981922477981571
F1 Weighted : 0.8808117346422094

Classification Report:
              precision    recall  f1-score   support

       angry       0.96      1.00      0.98        49
     disgust       0.85      0.94      0.89        64
        fear       0.83      0.72      0.77        60
       happy       0.97      0.97      0.97        35
     neutral       0.81      0.80      0.81        75
         sad       0.98      0.98      0.98        41

    accuracy                           0.88       324
   macro avg       0.90      0.90      0.90       324
weighted avg       0.88      0.88      0.88       324

Confusion Matrix:
[[49  0  0  0  0  0]
 [ 1 60  0  0  3  0]
 [ 0  6 43  0 10  1]
 [ 0  0  1 34  0  0]
 [ 1  5  8  1 60  0]
 [ 0  0  0  0  1 40]]

===== Predicted Emotion Distribution: Naive Bayes =====
   Emotion  Predicted_Count
0    angry               51
1  disgust          

### Load DL model

In [11]:
print("\nLoading DL model...")

with open(TOKENIZER_FILE, "r", encoding="utf-8") as f:
    tokenizer_json = f.read()
tokenizer = tokenizer_from_json(tokenizer_json)

with open(DL_METADATA_FILE, "r", encoding="utf-8") as f:
    dl_metadata = json.load(f)

max_seq = dl_metadata["max_sequence_length"]
classes = dl_metadata["classes"]

label_enc = LabelEncoder()
label_enc.classes_ = np.array(classes)

X_test_dl = tokenizer.texts_to_sequences(test_df["text"].tolist())
X_test_dl = pad_sequences(
    X_test_dl,
    maxlen=max_seq,
    padding="post",
    truncating="post"
)

dl_model = load_model(DL_MODEL_FILE)
dl_prob = dl_model.predict(X_test_dl, verbose=0)
dl_preds = label_enc.inverse_transform(np.argmax(dl_prob, axis=1))

result_df["dl_pred"] = dl_preds

print_model_result(y_true, dl_preds, "DL")
print_prediction_distribution(dl_preds, "DL")

summary_rows.append(get_summary_metrics(y_true, dl_preds, "DL"))
per_class_frames.append(get_per_class_report(y_true, dl_preds, "DL"))
confusion_frames.append(get_confusion_rows(y_true, dl_preds, "DL"))

hybrid_dl_preds = run_hybrid(
    texts=test_df["text"].tolist(),
    baseline_preds=dl_preds,
    tag="DL"
)
result_df["dl_qwen_pred"] = hybrid_dl_preds

print_model_result(y_true, hybrid_dl_preds, "DL + Qwen")
print_prediction_distribution(hybrid_dl_preds, "DL + Qwen")

summary_rows.append(get_summary_metrics(y_true, hybrid_dl_preds, "DL + Qwen"))
per_class_frames.append(get_per_class_report(y_true, hybrid_dl_preds, "DL + Qwen"))
confusion_frames.append(get_confusion_rows(y_true, hybrid_dl_preds, "DL + Qwen"))


Loading DL model...

===== DL =====
Accuracy    : 0.8950617283950617
F1 Macro    : 0.907828570979043
F1 Weighted : 0.8944072852700758

Classification Report:
              precision    recall  f1-score   support

       angry       0.96      1.00      0.98        49
     disgust       0.92      0.91      0.91        64
        fear       0.76      0.80      0.78        60
       happy       0.94      0.97      0.96        35
     neutral       0.88      0.80      0.84        75
         sad       0.95      1.00      0.98        41

    accuracy                           0.90       324
   macro avg       0.90      0.91      0.91       324
weighted avg       0.90      0.90      0.89       324

Confusion Matrix:
[[49  0  0  0  0  0]
 [ 0 58  4  1  0  1]
 [ 0  4 48  0  8  0]
 [ 0  0  1 34  0  0]
 [ 2  1 10  1 60  1]
 [ 0  0  0  0  0 41]]

===== Predicted Emotion Distribution: DL =====
   Emotion  Predicted_Count
0    angry               51
1  disgust               63
2     fear           

### Build final tables

In [12]:
summary_df = pd.DataFrame(summary_rows).sort_values(
    by=["Accuracy", "F1_Macro"],
    ascending=False
).reset_index(drop=True)

per_class_df = pd.concat(per_class_frames, ignore_index=True)
confusion_df = pd.concat(confusion_frames, ignore_index=True)

### Save all

In [13]:
result_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
summary_df.to_csv(SUMMARY_FILE, index=False, encoding="utf-8-sig")
per_class_df.to_csv(PER_CLASS_FILE, index=False, encoding="utf-8-sig")
confusion_df.to_csv(CONFUSION_FILE, index=False, encoding="utf-8-sig")

### Final console summary

In [14]:
print("\n================ FINAL SUMMARY TABLE ================")
print(summary_df)

print("\n================ PER-CLASS SUMMARY TABLE ================")
print(per_class_df)

best_model_name = summary_df.iloc[0]["Model"]
print(f"\nBest model from summary: {best_model_name}")

print_per_emotion_summary(per_class_df, best_model_name)

print("\nSaved:")
print("-", OUTPUT_FILE)
print("-", SUMMARY_FILE)
print("-", PER_CLASS_FILE)
print("-", CONFUSION_FILE)
print("Done")


================ FINAL SUMMARY TABLE ================
                           Model  Accuracy  F1_Macro  F1_Weighted
0         Support Vector Machine  0.898148  0.914325     0.897131
1            Logistic Regression  0.895062  0.912052     0.894449
2                             DL  0.895062  0.907829     0.894407
3                    Naive Bayes  0.882716  0.898192     0.880812
4                  Random Forest  0.861111  0.885660     0.863335
5  Support Vector Machine + Qwen  0.805556  0.815797     0.807874
6                      DL + Qwen  0.802469  0.813397     0.805275
7           Random Forest + Qwen  0.799383  0.811433     0.802352
8     Logistic Regression + Qwen  0.799383  0.810765     0.801793
9             Naive Bayes + Qwen  0.796296  0.806823     0.798463

================ PER-CLASS SUMMARY TABLE ================
                            Model  Emotion  Precision    Recall  F1_Score  Support
0                     Naive Bayes    angry   0.960784  1.000000  0.980000    